# Notebook 07 — ARMA–GARCH Baseline

## Purpose

This notebook builds the first formal volatility model.

- **ARMA** models the expected return or conditional mean.
- **GARCH** models the expected volatility or conditional variance.

The notebook compares normal and Student-t GARCH errors and checks whether volatility dependence remains after fitting.

In [1]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# Find the project root whether the notebook is launched from:
#   project/
# or:
#   project/notebooks/
CURRENT = Path.cwd().resolve()

if (CURRENT / "data").exists():
    PROJECT_ROOT = CURRENT
elif CURRENT.name == "notebooks" and (CURRENT.parent / "data").exists():
    PROJECT_ROOT = CURRENT.parent
else:
    possible_roots = [CURRENT, *CURRENT.parents]
    matches = [p for p in possible_roots if (p / "data").exists() and (p / "notebooks").exists()]
    if not matches:
        raise FileNotFoundError(
            "Could not find the project root. Open the sp500-forecasting-dissertation "
            "folder in VS Code, then run this notebook again."
        )
    PROJECT_ROOT = matches[0]

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
REPORT_TABLES = PROJECT_ROOT / "reports" / "tables"
REPORT_FIGURES = PROJECT_ROOT / "reports" / "figures"
MODEL_DIR = PROJECT_ROOT / "reports" / "models"

for folder in [DATA_PROCESSED, REPORT_TABLES, REPORT_FIGURES, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Python:", sys.executable)
print("Python version:", sys.version.split()[0])

Project root: <project_root>
Python: /opt/anaconda3/envs/dissertation/bin/python
Python version: 3.11.15


In [2]:
def find_crsp_panel(processed_folder: Path) -> Path:
    """Find the best available corrected CRSP parquet file."""
    preferred_names = [
        "crsp_sp500_daily_corrected_2010_2024.parquet",
        "crsp_sp500_daily_corrected.parquet",
        "crsp_daily_corrected_2010_2024.parquet",
        "crsp_daily_2010_2024.parquet",
    ]

    for name in preferred_names:
        path = processed_folder / name
        if path.exists():
            return path

    candidates = sorted(
        processed_folder.glob("*.parquet"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )

    if not candidates:
        raise FileNotFoundError(
            f"No parquet file was found in {processed_folder}. "
            "Run and save the corrected CRSP data notebook first."
        )

    print("No preferred filename found. Using the newest parquet file:")
    return candidates[0]


PANEL_PATH = find_crsp_panel(DATA_PROCESSED)
panel = pd.read_parquet(PANEL_PATH).copy()
panel.columns = [str(c).strip().lower() for c in panel.columns]

# Accept the possible names used in the earlier audit notebook.
return_candidates = ["ret", "combined_return", "ret_combined"]
return_source = next((c for c in return_candidates if c in panel.columns), None)

required_base = {"permno", "date"}
missing_base = required_base.difference(panel.columns)

if missing_base:
    raise ValueError(f"Missing required columns: {sorted(missing_base)}")

if return_source is None:
    raise ValueError(
        "Could not find a return column. Expected one of: "
        f"{return_candidates}. Found: {panel.columns.tolist()}"
    )

panel["date"] = pd.to_datetime(panel["date"])
panel["permno"] = pd.to_numeric(panel["permno"], errors="coerce").astype("Int64")
panel[return_source] = pd.to_numeric(panel[return_source], errors="coerce")

if "mktcap" in panel.columns:
    panel["mktcap"] = pd.to_numeric(panel["mktcap"], errors="coerce")

panel = (
    panel.dropna(subset=["permno", "date"])
         .sort_values(["date", "permno"])
         .reset_index(drop=True)
)

# IMPORTANT DECISION:
# "simple" keeps CRSP total returns, including a possible -100% delisting return.
# "log" uses log(1 + return), but an exact -100% return cannot be logged.
RETURN_MODE = "simple"

if RETURN_MODE == "simple":
    panel["model_return"] = panel[return_source]
elif RETURN_MODE == "log":
    impossible_for_log = panel[return_source] <= -1
    print("Rows not usable as log-returns:", int(impossible_for_log.sum()))
    panel["model_return"] = np.where(
        panel[return_source] > -1,
        np.log1p(panel[return_source]),
        np.nan,
    )
else:
    raise ValueError("RETURN_MODE must be 'simple' or 'log'.")

print("Loaded:", PANEL_PATH)
print("Rows:", len(panel))
print("Dates:", panel["date"].min(), "to", panel["date"].max())
print("Unique PERMNOs:", panel["permno"].nunique())
print("Return source:", return_source)
print("Return mode:", RETURN_MODE)
print("Missing modelling returns:", panel["model_return"].isna().sum())

Loaded: <project_root>/data/processed/crsp_sp500_daily_corrected_2010_2024.parquet
Rows: 1711517
Dates: 2010-01-04 00:00:00 to 2024-12-31 00:00:00
Unique PERMNOs: 740
Return source: ret
Return mode: simple
Missing modelling returns: 60


## Important context

A lower GARCH variance error does not automatically mean better return-direction forecasts.

GARCH is primarily useful because it tells us how risky or volatile the next period may be. That forecast can later be used for:

- volatility-scaled stock rankings;
- portfolio risk control;
- GARCH-in-mean extensions;
- more realistic uncertainty estimates.

In [3]:
# Choose one stock with a long and complete return history.

stock_counts = (
    panel.dropna(subset=["model_return"])
         .groupby("permno")
         .size()
         .sort_values(ascending=False)
)

example_permno = int(stock_counts.index[0])
example_rows = panel[panel["permno"] == example_permno].copy()

if "ticker" in example_rows.columns and example_rows["ticker"].notna().any():
    example_label = str(example_rows["ticker"].dropna().iloc[-1])
else:
    example_label = str(example_permno)

returns = (
    example_rows[["date", "model_return"]]
    .dropna()
    .sort_values("date")
    .set_index("date")["model_return"]
)

print("Example:", example_label, "| PERMNO:", example_permno)
print("Observations:", len(returns))

Example: ADI | PERMNO: 60871
Observations: 3774


## Step 1 — Select an ARMA order using BIC

**BIC is not a hypothesis test.**

It asks:

> Which model balances fit and simplicity?

Lower BIC is preferred because it penalises unnecessary parameters.

In [4]:
from itertools import product
from statsmodels.tsa.arima.model import ARIMA


def select_arma_by_bic(series: pd.Series, max_p: int = 3, max_q: int = 3):
    rows = []
    fitted_models = {}

    for p, q in product(range(max_p + 1), range(max_q + 1)):
        if p == 0 and q == 0:
            continue

        try:
            fitted = ARIMA(series, order=(p, 0, q)).fit()
            rows.append({
                "p": p,
                "q": q,
                "aic": fitted.aic,
                "bic": fitted.bic,
            })
            fitted_models[(p, q)] = fitted
        except Exception as error:
            rows.append({
                "p": p,
                "q": q,
                "aic": np.nan,
                "bic": np.nan,
            })

    table = pd.DataFrame(rows).sort_values("bic", na_position="last")
    valid = table.dropna(subset=["bic"])

    if valid.empty:
        raise RuntimeError("No ARMA candidate fitted successfully.")

    best_order = (int(valid.iloc[0]["p"]), int(valid.iloc[0]["q"]))
    return best_order, fitted_models[best_order], table


TRAIN_END = "2018-12-31"
train_returns = returns.loc[:TRAIN_END]

best_order, arma_result, arma_bic_table = select_arma_by_bic(
    train_returns,
    max_p=3,
    max_q=3,
)

print("Best ARMA order by BIC:", best_order)
arma_bic_table.head(10)

/opt/anaconda3/envs/dissertation/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/opt/anaconda3/envs/dissertation/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/opt/anaconda3/envs/dissertation/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/opt/anaconda3/envs/dissertation/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency in

Best ARMA order by BIC: (0, 1)


,p,q,aic,bic
0,0,1,-12337.633290,-12320.458625
3,1,0,-12337.489905,-12320.315239
4,1,1,-12339.128374,-12316.228820
1,0,2,-12336.451803,-12313.552249
7,2,0,-12336.228818,-12313.329264
2,0,3,-12336.293239,-12307.668797
11,3,0,-12335.879998,-12307.255556
5,1,2,-12334.400861,-12305.776419
8,2,1,-12334.275646,-12305.651204
9,2,2,-12339.816668,-12305.467337


## Step 2 — Test for ARCH effects before GARCH

The ARCH-LM null hypothesis is:

> There are no ARCH effects.

A small p-value supports using GARCH.

In [5]:
from statsmodels.stats.diagnostic import het_arch

arma_residuals = pd.Series(arma_result.resid).dropna()

arch_lm_before = het_arch(arma_residuals, nlags=10)

print("ARCH-LM statistic:", arch_lm_before[0])
print("ARCH-LM p-value:", arch_lm_before[1])

ARCH-LM statistic: 56.667928515352756
ARCH-LM p-value: 1.538675835486903e-08


## Step 3 — Fit GARCH(1,1)

We fit GARCH to the ARMA residuals.

The GARCH equation has three important parameters:

- **omega:** long-run variance base;
- **alpha:** reaction to yesterday's shock;
- **beta:** persistence of yesterday's volatility.

`alpha + beta` close to one means volatility shocks fade slowly.

In [6]:
from arch import arch_model

SCALE = 100.0
scaled_residuals = arma_residuals * SCALE

garch_results = {}

for distribution in ["normal", "t"]:
    model = arch_model(
        scaled_residuals,
        mean="Zero",
        vol="GARCH",
        p=1,
        q=1,
        dist=distribution,
        rescale=False,
    )

    fitted = model.fit(disp="off")
    garch_results[distribution] = fitted

comparison = pd.DataFrame({
    name: {
        "log_likelihood": result.loglikelihood,
        "aic": result.aic,
        "bic": result.bic,
    }
    for name, result in garch_results.items()
}).T

comparison

,log_likelihood,aic,bic
normal,-4174.799662,8355.599325,8372.773990
t,-4078.284358,8164.568716,8187.468269


## Why compare normal and Student-t errors?

Jarque–Bera often rejects normality for stock returns.

Student-t errors allow heavier tails, so they may fit extreme returns more realistically.

Again, lower AIC/BIC is better, but the final choice must also be checked out of sample.

In [7]:
best_distribution = comparison["bic"].idxmin()
best_garch = garch_results[best_distribution]

print("Best distribution by BIC:", best_distribution)
print(best_garch.summary())

Best distribution by BIC: t
                          Zero Mean - GARCH Model Results                           
Dep. Variable:                         None   R-squared:                       0.000
Mean Model:                       Zero Mean   Adj. R-squared:                  0.000
Vol Model:                            GARCH   Log-Likelihood:               -4078.28
Distribution:      Standardized Student's t   AIC:                           8164.57
Method:                  Maximum Likelihood   BIC:                           8187.47
                                              No. Observations:                 2264
Date:                      Fri, Jun 26 2026   Df Residuals:                     2264
Time:                              19:13:35   Df Model:                            0
                              Volatility Model                              
                 coef    std err          t      P>|t|      95.0% Conf. Int.
-----------------------------------------------------

## Step 4 — Interpret volatility persistence

For GARCH(1,1):

- alpha measures the immediate effect of a shock;
- beta measures persistence;
- alpha + beta measures total volatility persistence.

In [8]:
params = best_garch.params

alpha_name = next((name for name in params.index if "alpha" in name.lower()), None)
beta_name = next((name for name in params.index if "beta" in name.lower()), None)

alpha = float(params[alpha_name]) if alpha_name else np.nan
beta = float(params[beta_name]) if beta_name else np.nan
persistence = alpha + beta

print("Alpha:", alpha)
print("Beta:", beta)
print("Alpha + Beta:", persistence)

if np.isfinite(persistence):
    if persistence < 1:
        print("Interpretation: volatility is persistent but mean-reverting.")
    else:
        print("Warning: persistence is at or above one; investigate the specification.")

Alpha: 0.05616527704716723
Beta: 0.9238713281511337
Alpha + Beta: 0.9800366051983009
Interpretation: volatility is persistent but mean-reverting.


## Step 5 — Residual diagnostics after GARCH

A successful volatility model should leave:

- little autocorrelation in standardised residuals;
- little autocorrelation in squared standardised residuals;
- little remaining ARCH effect.

Jarque–Bera may still reject perfect normality. That is not automatically a failure.

In [9]:
from scipy import stats
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

std_resid = pd.Series(best_garch.std_resid).replace([np.inf, -np.inf], np.nan).dropna()

lb_resid = acorr_ljungbox(std_resid, lags=[10], return_df=True)
lb_squared = acorr_ljungbox(std_resid ** 2, lags=[10], return_df=True)
arch_after = het_arch(std_resid, nlags=10)
jb_after = stats.jarque_bera(std_resid)

diagnostics = pd.DataFrame({
    "diagnostic": [
        "Ljung-Box on standardised residuals",
        "Ljung-Box on squared standardised residuals",
        "ARCH-LM on standardised residuals",
        "Jarque-Bera on standardised residuals",
    ],
    "p_value": [
        float(lb_resid["lb_pvalue"].iloc[0]),
        float(lb_squared["lb_pvalue"].iloc[0]),
        float(arch_after[1]),
        float(jb_after.pvalue),
    ],
})

diagnostics["reject_null_at_5pct"] = diagnostics["p_value"] < 0.05
diagnostics

,diagnostic,p_value,reject_null_at_5pct
0,Ljung-Box on standardised residuals,2.724298e-01,False
1,Ljung-Box on squared standardised residuals,7.068741e-01,False
2,ARCH-LM on standardised residuals,7.296232e-01,False
3,Jarque-Bera on standardised residuals,3.345425e-132,True


## Step 6 — One-step-ahead forecast

- ARMA provides the mean-return forecast.
- GARCH provides the variance forecast.

In [10]:
arma_mean_forecast = float(arma_result.forecast(steps=1).iloc[0])

garch_forecast = best_garch.forecast(horizon=1, reindex=False)
variance_scaled = float(garch_forecast.variance.values[-1, 0])
variance_original = variance_scaled / (SCALE ** 2)
volatility_original = np.sqrt(variance_original)

print("Next return mean forecast:", arma_mean_forecast)
print("Next variance forecast:", variance_original)
print("Next volatility forecast:", volatility_original)

Next return mean forecast: 0.0005142771904064005
Next variance forecast: 0.00044716210089629867
Next volatility forecast: 0.021146207719028455


/opt/anaconda3/envs/dissertation/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(


In [11]:
comparison.to_csv(REPORT_TABLES / "garch_distribution_comparison.csv")
diagnostics.to_csv(REPORT_TABLES / "garch_residual_diagnostics.csv", index=False)
arma_bic_table.to_csv(REPORT_TABLES / "arma_bic_order_search.csv", index=False)

with open(REPORT_TABLES / "garch_model_summary.txt", "w") as file:
    file.write(str(best_garch.summary()))

print("Saved GARCH tables and summary.")

Saved GARCH tables and summary.


## What to say in the meeting

> The ARCH-LM test tells me whether volatility clustering is formally present. I first use ARMA to remove linear mean dependence, then fit GARCH to the remaining shocks. I compare normal and Student-t errors because Jarque–Bera indicates heavy tails. Finally, I run residual tests to check whether the model has actually removed the dependence it was designed to capture.